In [2]:
!pip install riskfolio-lib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.7/313.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.7/421.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
import riskfolio as rp
import joblib
from sklearn.covariance import LedoitWolf
import joblib
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')


# 1. ЗАГРУЗКА ДАННЫХ
fold = 3 # Укажите нужный фолд
base_path = f"/content/drive/MyDrive/my_trading_project/ensemble_storage/"
audit_path = f"/content/drive/MyDrive/my_trading_project/audit_reports/"

print("⏳ Загрузка файлов...")
df_test = pd.read_csv(f"{base_path}test.csv", parse_dates=['date'])
raw_signals = joblib.load(f"{base_path}raw_signals.joblib")
df_audit = pd.read_csv(f"{audit_path}df_audit_readable_fold_3.csv", parse_dates=['date'])
raw_signals = joblib.load(f"{base_path}raw_signals.joblib")

# 2. ВЫРАВНИВАНИЕ (Preprocessing)
# Приводим к единому индексу дат
df_test = df_test.sort_values(['date', 'tic'])
df_audit = df_audit.sort_values('date').set_index('date')

# Извлекаем цены закрытия для доходностей (Pivot table)
prices = df_test.pivot(index='date', columns='tic', values='close')
returns = prices.pct_change().dropna()

# Выравниваем аудит под даты доходностей
common_dates = returns.index.intersection(df_audit.index)
returns = returns.loc[common_dates]
df_audit = df_audit.loc[common_dates]

raw_signals = joblib.load(f"{base_path}raw_signals.joblib")
# Если raw_signals — это словарь {'ppo': df_actions}, достаем DataFrame
if isinstance(raw_signals, dict) and 'ppo' in raw_signals:
    df_actions = raw_signals['ppo']
else:
    df_actions = raw_signals

#print(df_actions)
print(f"✅ Синхронизация завершена: {len(common_dates)} торговых дней.")



Mounted at /content/drive
⏳ Загрузка файлов...
✅ Синхронизация завершена: 533 торговых дней.


In [23]:
import pandas as pd
import numpy as np
import warnings
from sklearn.covariance import LedoitWolf
import riskfolio as rp
from numpy.linalg import LinAlgError

class BlackLittermanPipeline:
    def __init__(self, tau=0.05, conf_base=0.1, short_limit=-0.5, min_data_points=30):
        self.tau = tau
        self.conf_base = conf_base
        self.short_limit = short_limit
        self.min_data_points = min_data_points

    def _validate_data(self, df_returns_window):
        if df_returns_window.empty:
            return False, "Empty returns data"
        if len(df_returns_window) < self.min_data_points:
            return False, f"Insufficient data: {len(df_returns_window)}"

        # Очистка данных
        df_returns_window = df_returns_window.fillna(df_returns_window.mean())
        if np.isinf(df_returns_window).any().any():
            return False, "Infinite values"
        return True, df_returns_window

    def _safe_matrix_inversion(self, matrix, epsilon=1e-8):
        try:
            reg_matrix = matrix + np.eye(matrix.shape[0]) * epsilon
            return np.linalg.inv(reg_matrix)
        except LinAlgError:
            return np.linalg.pinv(matrix)

    def clean_weights(self, weights, threshold=0.01):
        """
        Убирает веса меньше 1%, пересчитывает остальные, чтобы сумма была 100%.
        """
        try:
            # 1. Обнуляем все, что меньше порога (0.01 = 1%)
            clean_w = weights.copy()
            clean_w[clean_w < threshold] = 0

            # 2. Нормализуем, чтобы сумма весов снова была равна 1.0 (100%)
            if clean_w.sum() > 0:
                clean_w = clean_w / clean_w.sum()

            # 3. Форматируем для вывода в %
            return clean_w.apply(lambda x: f"{x:.2%}")
        except Exception as e:
            print(f"Pipeline function clean_weights error: {e}")

    def prepare_input_data(self, df_test, df_audit, window_size=50):
        """
        Исправленная подготовка данных: корректная обработка индекса даты и тикеров.
        """
        try:
            # 1. Подготовка доходностей
            prices = df_test.pivot(index='date', columns='tic', values='close')
            prices.index = pd.to_datetime(prices.index)
            returns = prices.pct_change().dropna()
            rets_50 = returns.tail(window_size)

            # 2. Подготовка audit_row (учитываем, что date может быть в индексе)
            audit_df = df_audit.copy()
            if 'date' in audit_df.columns:
                audit_df['date'] = pd.to_datetime(audit_df['date'])
                audit_df = audit_df.set_index('date')
            else:
                audit_df.index = pd.to_datetime(audit_df.index)

            audit_df = audit_df.sort_index()
            audit_last_row = audit_df.iloc[-1]

            # 3. Генерация сигналов PPO (строго под тикеры из rets_50)
            current_tickers = rets_50.columns
            current_date = rets_50.index[-1]

            # Генерируем разные mock-сигналы для каждого инструмента (от 0.2 до 0.8)
            np.random.seed(42) # Для воспроизводимости mock-данных
            mock_values = np.random.uniform(0.2, 0.8, size=len(current_tickers))
            ppo_row = pd.Series(mock_values, index=current_tickers, name=current_date)

            return rets_50, audit_last_row, ppo_row
        except Exception as e:
            print(f"Pipeline function prepare_input_data error: {e}")
            return pd.Series(0, index=df_test.columns)

    def generate_weights(self, df_returns_window, audit_row, ppo_action_row):
        try:
            df_test = df_returns_window.copy()
            df_audit = audit_row.copy()
            if len(ppo_action_row) < 10:
              df_returns_window, audit_row, ppo_action_row = self.prepare_input_data(df_test, df_audit)
            is_valid, result = self._validate_data(df_returns_window)
            if not is_valid:
                return pd.Series(0, index=ppo_action_row.index)
            df_returns_window = result

            # Оценка параметров
            lw = LedoitWolf().fit(df_returns_window)
            cov_pd = pd.DataFrame(lw.covariance_, index=df_returns_window.columns, columns=df_returns_window.columns)
            mu_eq = df_returns_window.mean()

            # Views (PPO)
            views_mu = ((ppo_action_row - 0.5) * 0.05).clip(-0.2, 0.2)
            P = np.eye(len(df_returns_window.columns))
            Q = views_mu.values

            # Omega (Уверенность)
            diag_omega = []
            conf = audit_row.get('confidence', self.conf_base) + 1e-6
            kurt = audit_row.get('kurtosis', 3)
            for tic in df_returns_window.columns:
                r_val = audit_row.get(f'risk_{tic}', audit_row.get('tail_risk', 0.5))
                unc = (r_val / conf) * 0.02 + 1e-8
                if kurt > 1.15: unc *= 2
                diag_omega.append(unc)
            Omega = np.diag(diag_omega)

            # BL Math
            M_inv = self._safe_matrix_inversion(self.tau * cov_pd.values)
            O_inv = self._safe_matrix_inversion(Omega)
            mu_bl = np.linalg.solve(M_inv + P.T @ O_inv @ P, M_inv @ mu_eq.values + P.T @ O_inv @ Q)
            mu_bl_series = pd.Series(mu_bl, index=df_returns_window.columns)

            # Оптимизация
            port = rp.Portfolio(returns=df_returns_window)
            port.mu, port.cov = mu_bl_series, cov_pd

            # Настройка режима кризиса
            is_crisis = (kurt > 1.15 or audit_row.get('tail_risk', 0.5) > 0.6 or conf < 0.05)

            port.allow_shorts = True
            port.lower_short = self.short_limit
            port.upper_short = 0.5
            port.budget = 1.0
            port.upperbound = 0.3 if is_crisis else 0.7

            w = port.optimization(model='Classic', rm='CVaR' if is_crisis else 'MV',
                                  obj='MinRisk' if is_crisis else 'Sharpe', rf=0, hist=True)

            if w is None or w.empty:
                return pd.Series(1/len(df_returns_window.columns), index=df_returns_window.columns)

            w['weights'] = self.clean_weights(w['weights'])
            # Возвращаем веса как есть (Riskfolio уже применил budget=1.0)
            return w['weights']

        except Exception as e:
            print(f"Pipeline Error: {e}")
            return pd.Series(0, index=df_returns_window.columns)



In [24]:
import pandas as pd
import numpy as np
from sklearn.covariance import LedoitWolf
import riskfolio as rp

class TradingExecutionPipeline:
    def __init__(self, bl_params=None, lot_sizes=None):
        # Параметры Black-Litterman
        self.bl_params = bl_params or {
            'tau': 0.05,
            'conf_base': 0.1,
            'short_limit': -0.5,
            'min_data_points': 30
        }
        # Инициализируем расчетный модуль BL
        self.bl_engine = BlackLittermanPipeline(**self.bl_params)

        # Справочник лотов
        self.lot_sizes = lot_sizes or {
            'SBER_USD': 10, 'SBRF_USD': 10, 'OFZ_FIX': 1,
            'GOLD_USD': 1, 'GOLD_INV': 1, 'SBER_INV': 1
        }

    def execute_cycle(self, df_returns, audit_row, ppo_signals, current_balances, current_prices, total_account):
        """
        Полный цикл: Данные -> Веса BL -> Ордера -> Лоты
        """
        # 1. Генерация целевых весов через Black-Litterman
        target_weights = self.bl_engine.generate_weights(
            df_returns, audit_row, ppo_signals
        )

        if target_weights.sum() == 0 and not target_weights.empty:
            print("Предупреждение: Получены нулевые веса. Ребалансировка пропущена.")
            return None
        # print("Сырые веса от оптимизатора:", target_weights)
        # 2. Расчет базовых ордеров (с учетом текущих позиций и Long/Short)
        orders_df = self._calculate_raw_orders(
            target_weights, current_balances, current_prices, total_account
        )

        # 3. Применение кратности лотов
        final_orders = self._apply_lot_constraints(orders_df)

        # 4. Формирование финального плана действий для робота
        trading_plan = self._generate_action_plan(final_orders)

        return trading_plan

    def _calculate_raw_orders(self, tw, balances, prices, total_cash):
        """Внутренний метод расчета дельты позиций"""
        # 1. Принудительное приведение к числам (float)
        # errors='coerce' превратит строки, которые нельзя сконвертировать, в NaN
        # Очистка весов: убираем '%' и конвертируем в float
        if tw.dtype == 'object':
            tw = tw.str.replace('%', '', regex=False).astype(float) / 100
        else:
            tw = pd.to_numeric(tw, errors='coerce').fillna(0)

        tw = pd.to_numeric(tw, errors='coerce').fillna(0)

        # 2. Убираем шум (меньше 0.5%), сохраняя знаки для Long/Short
        tw = tw.copy()
        tw[tw.abs() < 0.005] = 0

        orders = []
        for tic in tw.index:
            price = prices.get(tic, 0)
            if price <= 0: continue

            target_val = total_cash * tw[tic]
            current_qty = balances.get(tic, 0)
            current_val = current_qty * price

            delta_val = target_val - current_val
            # Используем отсечение дробной части (trunc) вместо округления
            delta_qty = int(np.trunc(delta_val / price))

            orders.append({
                'ticker': tic,
                'target_weight': tw[tic],
                'current_qty': current_qty,
                'order_qty': delta_qty,
                'order_value': delta_val
            })
        return pd.DataFrame(orders).set_index('ticker')

    def _apply_lot_constraints(self, df):
        """Корректировка с учетом лотов"""
        df = df.copy()
        df['lot_size'] = df.index.map(lambda x: self.lot_sizes.get(x, 1))

        # Округляем количество в сторону нуля до ближайшего целого лота
        df['order_qty_lots'] = (np.trunc(df['order_qty'] / df['lot_size']) * df['lot_size']).astype(int)

        return df[df['order_qty_lots'] != 0]

    def _generate_action_plan(self, df):
        """Создание читаемых команд для робота"""
        if df.empty: return pd.DataFrame()

        plan = df.copy()
        plan['Action'] = plan['order_qty_lots'].apply(
            lambda x: 'BUY/COVER' if x > 0 else 'SELL/SHORT'
        )
        plan['Qty_to_Trade'] = plan['order_qty_lots'].abs()

        # Форматирование для вывода
        plan['target_weight'] = plan['target_weight'].apply(lambda x: f"{x:.2%}")
        plan['order_value'] = plan['order_value'].round(2)

        return plan[['Action', 'target_weight', 'Qty_to_Trade', 'order_value']].sort_values(by='Action')

bot_pipeline = TradingExecutionPipeline(lot_sizes={'SBER_USD': 10, 'GOLD_USD': 1})
ready_plan = bot_pipeline.execute_cycle(
     df_returns=df_test,
     audit_row=df_audit,
     ppo_signals=df_actions,
     current_balances={'SBER_USD': 100, 'GOLD_USD': 10, 'SBRF_USD': 7}, # текущие активы
     current_prices= df_test[df_test['date'] == df_test['date'].max()].set_index('tic')['close'],         # текущие цены
     total_account=1000000               # кэш + оценка бумаг
 )
print(ready_plan)


              Action target_weight  Qty_to_Trade  order_value
ticker                                                       
GOLD_USD   BUY/COVER        12.20%           739    120372.17
OFZ_INV    BUY/COVER        56.16%        585665    561600.00
SBER_USD   BUY/COVER        25.89%         65330    258504.31
SBRF_INV   BUY/COVER         5.75%           214     57500.00
SBRF_USD  SELL/SHORT         0.00%             7     -2775.60
